# GRU Seq2Seq + Bahdanau Attention smoke test (3 epochs)

This notebook checks that the dataset and training pipeline work. Three epochs are only a smoke test, not a meaningful final training run.

這份 Notebook 用來確認資料集與訓練流程能正常運作。3 個 epoch 只是冒煙測試，不代表正式模型效果。

In [ ]:
# Run once if packages are missing / 缺少套件時執行一次
# %pip install torch numpy


In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_DIR = Path(r"C:\Users\User\Documents\Push-Github\sign-language\rewrite_Phase3")
LABEL_PATH = PROJECT_DIR / "Labels.json"
FEATURE_DIR = PROJECT_DIR / "features"
if not FEATURE_DIR.is_dir():
    FEATURE_DIR = PROJECT_DIR

EPOCHS = 3
BATCH_SIZE = 2
FRAME_STRIDE = 2  # use every second frame to reduce sequence length
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device / 裝置:", DEVICE)
print("Labels / 標籤:", LABEL_PATH)
print("Features / 特徵:", FEATURE_DIR)


In [ ]:
def normalize_records(raw):
    if isinstance(raw, dict) and "labels" in raw:
        raw = raw["labels"]
    if isinstance(raw, dict):
        raw = [{"npz_file": key, "text": value} for key, value in raw.items()]
    if not isinstance(raw, list):
        raise ValueError("Labels.json must be a list, mapping, or contain a 'labels' list")

    records = []
    for index, item in enumerate(raw):
        if not isinstance(item, dict):
            raise ValueError(f"Label item {index} is not an object")
        filename = item.get("npz_file") or item.get("file") or item.get("path")
        text = item.get("text") or item.get("subtitle") or item.get("label")
        if not filename or not isinstance(text, str) or not text.strip():
            raise ValueError(f"Label item {index} needs npz_file and non-empty text")
        npz_path = Path(filename)
        if not npz_path.is_absolute():
            npz_path = FEATURE_DIR / npz_path
        records.append({"npz_path": npz_path, "text": text.strip()})
    return records

with LABEL_PATH.open("r", encoding="utf-8-sig") as file:
    records = normalize_records(json.load(file))

missing = [str(item["npz_path"]) for item in records if not item["npz_path"].is_file()]
if missing:
    raise FileNotFoundError("Missing NPZ files / 找不到 NPZ:\n" + "\n".join(missing[:20]))

print("Samples / 樣本數:", len(records))
print("First sample / 第一筆:", records[0])
if len(records) < 10:
    print("WARNING: This is enough only for a pipeline test, not useful training.")


In [ ]:
PAD, BOS, EOS, UNK = "<pad>", "<bos>", "<eos>", "<unk>"
characters = sorted(set("".join(item["text"] for item in records)))
itos = [PAD, BOS, EOS, UNK] + characters
stoi = {token: index for index, token in enumerate(itos)}
PAD_ID, BOS_ID, EOS_ID, UNK_ID = (stoi[PAD], stoi[BOS], stoi[EOS], stoi[UNK])

def encode_text(text):
    return [BOS_ID] + [stoi.get(char, UNK_ID) for char in text] + [EOS_ID]

print("Vocabulary size / 字元表大小:", len(itos))


In [ ]:
def load_npz_features(npz_path):
    with np.load(npz_path) as data:
        hands = data["hands"].reshape(data["hands"].shape[0], -1)
        pose = data["pose"].reshape(data["pose"].shape[0], -1)
        hand_mask = data["hand_mask"].reshape(data["hand_mask"].shape[0], -1)
        pose_mask = data["pose_mask"].reshape(data["pose_mask"].shape[0], -1)
        confidence = data["pose_confidence"]
        features = np.concatenate(
            [hands, pose, hand_mask, pose_mask, confidence], axis=1
        ).astype(np.float32)
    return features[::FRAME_STRIDE]

class SignDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        item = self.items[index]
        features = torch.from_numpy(load_npz_features(item["npz_path"]))
        tokens = torch.tensor(encode_text(item["text"]), dtype=torch.long)
        return features, tokens

def collate_batch(batch):
    features, tokens = zip(*batch)
    lengths = torch.tensor([len(value) for value in features], dtype=torch.long)
    features = pad_sequence(features, batch_first=True, padding_value=0.0)
    token_inputs = pad_sequence(
        [value[:-1] for value in tokens], batch_first=True, padding_value=PAD_ID
    )
    token_targets = pad_sequence(
        [value[1:] for value in tokens], batch_first=True, padding_value=PAD_ID
    )
    return features, lengths, token_inputs, token_targets

dataset = SignDataset(records)
loader = DataLoader(
    dataset, batch_size=min(BATCH_SIZE, len(dataset)), shuffle=True, collate_fn=collate_batch
)
sample_features, sample_tokens = dataset[0]
print("Feature shape / 特徵形狀:", sample_features.shape)
print("Token shape / 文字形狀:", sample_tokens.shape)


In [ ]:
class AttentionGRUTranslator(nn.Module):
    def __init__(self, input_size, vocab_size, hidden_size=256, embedding_size=128):
        super().__init__()
        self.input_projection = nn.Sequential(
            nn.Linear(input_size, embedding_size),
            nn.LayerNorm(embedding_size),
            nn.ReLU(),
        )
        self.encoder = nn.GRU(embedding_size, hidden_size, batch_first=True)
        self.embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=PAD_ID)

        # Bahdanau additive attention.
        self.encoder_key = nn.Linear(hidden_size, hidden_size, bias=False)
        self.decoder_query = nn.Linear(hidden_size, hidden_size, bias=False)
        self.attention_score = nn.Linear(hidden_size, 1, bias=False)

        self.decoder = nn.GRU(
            embedding_size + hidden_size, hidden_size, batch_first=True
        )
        self.output = nn.Linear(hidden_size * 2, vocab_size)

    def encode(self, features, lengths):
        projected = self.input_projection(features)
        packed = pack_padded_sequence(
            projected, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_outputs, hidden = self.encoder(packed)
        encoder_outputs, _ = pad_packed_sequence(
            packed_outputs, batch_first=True, total_length=features.shape[1]
        )
        positions = torch.arange(features.shape[1], device=features.device).unsqueeze(0)
        source_mask = positions < lengths.to(features.device).unsqueeze(1)
        return encoder_outputs, hidden, source_mask

    def attend(self, hidden, encoder_outputs, source_mask):
        query = self.decoder_query(hidden[-1]).unsqueeze(1)
        keys = self.encoder_key(encoder_outputs)
        scores = self.attention_score(torch.tanh(keys + query)).squeeze(-1)
        scores = scores.masked_fill(~source_mask, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, weights

    def decode_step(self, token, hidden, encoder_outputs, source_mask):
        embedded = self.embedding(token).unsqueeze(1)
        context, weights = self.attend(hidden, encoder_outputs, source_mask)
        decoder_input = torch.cat([embedded, context.unsqueeze(1)], dim=-1)
        decoded, hidden = self.decoder(decoder_input, hidden)
        combined = torch.cat([decoded.squeeze(1), context], dim=-1)
        logits = self.output(combined)
        return logits, hidden, weights

    def forward(self, features, lengths, token_inputs):
        encoder_outputs, hidden, source_mask = self.encode(features, lengths)
        logits = []
        for step in range(token_inputs.shape[1]):
            step_logits, hidden, _ = self.decode_step(
                token_inputs[:, step], hidden, encoder_outputs, source_mask
            )
            logits.append(step_logits)
        return torch.stack(logits, dim=1)

model = AttentionGRUTranslator(
    input_size=sample_features.shape[1], vocab_size=len(itos)
).to(DEVICE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
print(model)


In [ ]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for features, lengths, token_inputs, token_targets in loader:
        features = features.to(DEVICE)
        token_inputs = token_inputs.to(DEVICE)
        token_targets = token_targets.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(features, lengths, token_inputs)
        loss = criterion(logits.reshape(-1, logits.shape[-1]), token_targets.reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()

    average_loss = total_loss / len(loader)
    print(f"Epoch {epoch}/{EPOCHS} - loss: {average_loss:.4f}")


In [ ]:
checkpoint_path = PROJECT_DIR / "smoke_test_gru_attention.pt"
torch.save(
    {
        "model_type": "gru_seq2seq_bahdanau_attention",
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "itos": itos,
        "stoi": stoi,
        "input_size": sample_features.shape[1],
        "frame_stride": FRAME_STRIDE,
        "epochs": EPOCHS,
    },
    checkpoint_path,
)
print("Saved / 已儲存:", checkpoint_path)


In [ ]:
def greedy_decode(model, features, max_output_length=200):
    model.eval()
    features = features.unsqueeze(0).to(DEVICE)
    lengths = torch.tensor([features.shape[1]], dtype=torch.long)

    with torch.no_grad():
        encoder_outputs, hidden, source_mask = model.encode(features, lengths)
        current_token = torch.tensor([BOS_ID], device=DEVICE)
        generated_ids = []

        for _ in range(max_output_length):
            logits, hidden, _ = model.decode_step(
                current_token, hidden, encoder_outputs, source_mask
            )
            next_id = int(logits.argmax(dim=-1).item())

            if next_id == EOS_ID:
                break
            if next_id not in {PAD_ID, BOS_ID}:
                generated_ids.append(next_id)
            current_token = torch.tensor([next_id], device=DEVICE)

    return "".join(itos[index] for index in generated_ids)

sample_index = 0
sample_features, _ = dataset[sample_index]
prediction = greedy_decode(model, sample_features)
target = records[sample_index]["text"]

print("Target / 正確字幕:")
print(target)
print("\nPrediction / 模型預測:")
print(prediction if prediction else "<empty / 空白輸出>")


## Expected result / 預期結果

The loss should be a finite number and preferably decrease over the three epochs. With only a few samples, this confirms code execution but does not measure translation quality.

Loss 應該是有限數值，並最好在 3 個 epoch 中下降。資料很少時，這只能證明程式能執行，不能代表翻譯品質。